In [2]:
import plotly.graph_objects as go
import numpy as np

# --- データの準備 (G7 + China, 2023-2024 GDP) ---
# 色設定:
# CHN(中国) = 赤 (Red)
# JPN(日本) = 白/銀 (Silver) - 中国の赤と区別し、国旗の地色かつ「極東の技術ハブ」を表現
# DEU(ドイツ) = 緑 (Green)
g7_data = {
    "USA": {"coords": (38.90, -77.03), "gdp": 27.36, "rgb": (31, 119, 180)},   # Blue
    "CHN": {"coords": (39.90, 116.40), "gdp": 17.79, "rgb": (214, 39, 40)},    # Red (New!)
    "DEU": {"coords": (52.52, 13.40),  "gdp": 4.46,  "rgb": (44, 160, 44)},    # Green
    "JPN": {"coords": (35.67, 139.65), "gdp": 4.21,  "rgb": (220, 220, 220)},  # Silver (Changed from Red)
    "GBR": {"coords": (51.50, -0.12),  "gdp": 3.34,  "rgb": (148, 103, 189)},  # Purple
    "FRA": {"coords": (48.85, 2.35),   "gdp": 3.03,  "rgb": (0, 0, 128)},      # Navy
    "ITA": {"coords": (41.90, 12.49),  "gdp": 2.25,  "rgb": (188, 189, 34)},   # Olive
    "CAN": {"coords": (45.42, -75.69), "gdp": 2.14,  "rgb": (255, 127, 14)}    # Orange
}

def latlon_to_xyz(lat, lon):
    phi = np.radians(90 - lat)
    theta = np.radians(lon)
    return np.array([np.sin(phi) * np.cos(theta), np.sin(phi) * np.sin(theta), np.cos(phi)])

# 1. 解像度の設定 (論文用にやや細かく)
u_res, v_res = 200, 100
u = np.linspace(0, 2 * np.pi, u_res)
v = np.linspace(0, np.pi, v_res)
x = np.outer(np.cos(u), np.sin(v))
y = np.outer(np.sin(u), np.sin(v))
z = np.outer(np.ones(np.size(u)), np.cos(v))

# 2. テッセレーション計算 (多体問題の幾何学的解)
dominant_map = np.zeros(x.shape)
min_weighted_dist = np.full(x.shape, np.inf)
country_names = list(g7_data.keys())
# Fix: Use 'rgb' key instead of 'color' and store as rgb_colors
country_rgb_colors = [d["rgb"] for d in g7_data.values()]

for i, name in enumerate(country_names):
    d = g7_data[name]
    c_xyz = latlon_to_xyz(*d["coords"])
    # 測地線距離
    dist = np.arccos(np.clip(x * c_xyz[0] + y * c_xyz[1] + z * c_xyz[2], -1, 1))
    # 自律重み付け (カナダを消失させないための指数 0.12)
    weighted_dist = dist / (d["gdp"] ** 0.12)
    mask = weighted_dist < min_weighted_dist
    min_weighted_dist[mask] = weighted_dist[mask]
    dominant_map[mask] = i

# 3. 境界線（断層）の抽出
# 隣り合うセルでIDが変わる場所を特定する
boundary_x, boundary_y, boundary_z = [], [], []
for i in range(u_res - 1):
    for j in range(v_res - 1):
        # 横方向または縦方向で国が変わる場合、そこを境界点とする
        if dominant_map[i, j] != dominant_map[i+1, j] or dominant_map[i, j] != dominant_map[i, j+1]:
            # 球面からわずかに浮かせて(1.01)描画
            boundary_x.append(x[i,j] * 1.01)
            boundary_y.append(y[i,j] * 1.01)
            boundary_z.append(z[i,j] * 1.01)

# --- 描画 ---
fig = go.Figure()

# 球面ポリゴン (テッセレーション)
fig.add_trace(go.Surface(
    x=x, y=y, z=z,
    surfacecolor=dominant_map,
    # Fix: Convert RGB tuples to 'rgb(R, G, B)' strings for colorscale
    colorscale=[[i/(len(country_names)-1), f"rgb({rgb[0]}, {rgb[1]}, {rgb[2]})"] for i, rgb in enumerate(country_rgb_colors)],
    showscale=False, opacity=1.0,
    lighting=dict(ambient=0.6, diffuse=0.8),
    name="Tessellation"
))

# 境界線（Fault Lines）
# Scatter3dのマーカーモードを使って「点」を繋げて「線」に見せる手法（エラー回避）
fig.add_trace(go.Scatter3d(
    x=boundary_x, y=boundary_y, z=boundary_z,
    mode='markers',
    marker=dict(size=1.5, color='white', opacity=0.8),
    name="Fault Lines",
    showlegend=False
))

# 凡例用のダミープロット (右側のリスト)
for name, d in g7_data.items():
    c_xyz = latlon_to_xyz(*d["coords"])
    fig.add_trace(go.Scatter3d(
        x=[c_xyz[0]*1.05], y=[c_xyz[1]*1.05], z=[c_xyz[2]*1.05],
        mode='markers+text',
        marker=dict(size=5, color='white', line=dict(width=1, color='black')),
        text=[f"<b>{name}</b>"],
        textfont=dict(size=12, color="white"),
        textposition="top center",
        name=f"{name} ({d['gdp']}T)"
    ))

# レイアウト
fig.update_layout(
    title=dict(
        text="<b>Topological Global Economics</b><br>Structural Equilibrium and Economic Fault Lines",
        x=0.5, font=dict(size=20, color="white")
    ),
    scene=dict(
        xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
        aspectmode='data', bgcolor='rgb(10,10,10)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
    ),
    width=1000, height=800, margin=dict(r=0, l=0, b=0, t=60),
    paper_bgcolor='rgb(10,10,10)',
    showlegend=True,
    legend=dict(font=dict(color="white"), x=0.8, y=0.5)
)

fig.show()